# 🤖 Resume / Candidate Screening System
### FUTURE_ML_03 — Machine Learning Track

**Tools:** Python · spaCy / NLTK · Scikit-learn  
**Skills:** Text analysis · Feature extraction · Resume scoring · Ranking models

---
## Notebook Overview
| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Text Cleaning & Parsing |
| 3 | Skill Extraction & Matching |
| 4 | Scoring Engine (TF-IDF + Skill Match + Experience) |
| 5 | Candidate Ranking |
| 6 | Skill Gap Analysis |
| 7 | Visualisations |
| 8 | Interactive Demo |


## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install scikit-learn pandas numpy matplotlib seaborn spacy
# !python -m spacy download en_core_web_sm

import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Optional NLP backends ─────────────────────────────────────────────────────
NLP_BACKEND = "basic"
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    NLP_BACKEND = "spacy"
    print("✅ spaCy loaded successfully")
except Exception:
    try:
        import nltk
        for pkg in ["punkt", "stopwords", "averaged_perceptron_tagger", "wordnet"]:
            nltk.download(pkg, quiet=True)
        NLP_BACKEND = "nltk"
        print("✅ NLTK loaded successfully")
    except Exception:
        print("ℹ️  Using basic regex-based NLP (no spaCy/NLTK)")

print(f"NLP Backend: {NLP_BACKEND}")
pd.set_option("display.max_colwidth", 80)


## 2. Text Cleaning & Resume Parsing
We clean raw resume text, split it into sections, and extract key fields.

In [ ]:
class ResumeParser:
    """Clean and extract structured information from raw resume text."""

    SECTION_HEADERS = {
        "skills":         r"(skills?|technical\s+skills?|core\s+competencies|proficiencies)",
        "experience":     r"(experience|work\s+history|employment|professional\s+background)",
        "education":      r"(education|academic|qualification|degree)",
        "projects":       r"(projects?|portfolio|work\s+samples?)",
        "certifications": r"(certif|license|credential|course)",
        "summary":        r"(summary|objective|profile|about)",
    }

    SKILL_PATTERNS = [
        r"\b(python|java|javascript|typescript|c\+\+|c#|go|rust|ruby|php|swift|kotlin|scala|r|matlab)\b",
        r"\b(react|angular|vue|django|flask|fastapi|spring|node\.?js|express|laravel|rails)\b",
        r"\b(tensorflow|pytorch|keras|scikit[\-\s]?learn|pandas|numpy|scipy|matplotlib|seaborn|"
        r"opencv|nltk|spacy|huggingface|transformers|xgboost|lightgbm|sklearn)\b",
        r"\b(aws|azure|gcp|docker|kubernetes|jenkins|terraform|ansible|ci/cd|devops|mlops)\b",
        r"\b(sql|mysql|postgresql|mongodb|redis|cassandra|elasticsearch|oracle|sqlite|neo4j)\b",
        r"\b(git|github|gitlab|jira|confluence|tableau|power\s?bi|excel|spark|hadoop|kafka)\b",
        r"\b(machine\s+learning|deep\s+learning|nlp|computer\s+vision|data\s+science|"
        r"statistics|algorithms?|data\s+structures?|oop|agile|scrum|rest|api|microservices)\b",
    ]

    def __init__(self):
        self.skill_regex = re.compile("|".join(self.SKILL_PATTERNS), re.IGNORECASE)

    def clean_text(self, text):
        text = re.sub(r"[^\x00-\x7F]+", " ", text)
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"[^\w\s\.\,\-\+\#\/]", " ", text)
        return text.strip()

    def extract_sections(self, text):
        sections = defaultdict(str)
        lines = text.split("\n")
        current_section = "general"
        for line in lines:
            line_lower = line.lower().strip()
            matched = False
            for sec_name, pattern in self.SECTION_HEADERS.items():
                if re.search(pattern, line_lower):
                    current_section = sec_name
                    matched = True
                    break
            if not matched:
                sections[current_section] += " " + line
        return dict(sections)

    def extract_skills(self, text):
        matches = self.skill_regex.findall(text.lower())
        skills = []
        for m in matches:
            if isinstance(m, tuple):
                skills.extend([s for s in m if s])
            else:
                skills.append(m)
        skills = [re.sub(r"\s+", " ", s.strip()) for s in skills]
        return sorted(set(skills))

    def extract_experience_years(self, text):
        patterns = [
            r"(\d+)\+?\s*years?\s+of\s+experience",
            r"(\d+)\+?\s*years?\s+experience",
        ]
        years = []
        for pat in patterns:
            for m in re.finditer(pat, text, re.IGNORECASE):
                years.append(float(m.group(1)))
        date_ranges = re.findall(
            r"\b(20\d{2}|19\d{2})\s*[-–—to]+\s*(20\d{2}|19\d{2}|present|current|now)\b",
            text, re.IGNORECASE
        )
        current_year = 2024
        for start, end in date_ranges:
            try:
                s = int(start)
                e = current_year if end.lower() in ("present","current","now") else int(end)
                years.append(max(0, e - s))
            except ValueError:
                pass
        return round(sum(years) / max(len(years), 1), 1) if years else 0.0

    def parse(self, text, name="Candidate"):
        clean = self.clean_text(text)
        sections = self.extract_sections(clean)
        skills = self.extract_skills(clean)
        exp_years = self.extract_experience_years(clean)
        return {
            "name": name,
            "raw_text": text,
            "clean_text": clean,
            "sections": sections,
            "skills": skills,
            "experience_years": exp_years,
            "word_count": len(clean.split()),
        }

parser = ResumeParser()
print("✅ ResumeParser ready")


## 3. Sample Data — Job Descriptions & Resumes

In [ ]:
# ── Job Description ──────────────────────────────────────────────────────────
JOB = {
    "title": "Senior Data Scientist",
    "description": """
    We are looking for a Senior Data Scientist to join our AI team.
    You will build machine learning models, perform statistical analysis,
    and deploy scalable ML pipelines. The ideal candidate has strong Python
    skills, experience with deep learning frameworks, and the ability to
    communicate complex results to stakeholders.
    Responsibilities:
    - Develop and deploy machine learning and deep learning models
    - Perform exploratory data analysis and feature engineering
    - Build NLP pipelines for text classification and entity extraction
    - Collaborate with engineering to deploy models via REST APIs
    Requirements:
    - 4+ years of experience in data science or machine learning
    - Proficiency in Python, scikit-learn, TensorFlow or PyTorch
    - Strong knowledge of statistics and algorithms
    - Experience with SQL and data manipulation using Pandas
    - Familiarity with cloud platforms (AWS, GCP, or Azure)
    """,
    "required_skills": [
        "python", "machine learning", "scikit-learn", "deep learning",
        "tensorflow", "pytorch", "statistics", "sql", "pandas", "numpy",
    ],
    "preferred_skills": [
        "nlp", "aws", "docker", "spark", "huggingface", "mlops",
        "kubernetes", "git", "flask", "fastapi",
    ],
    "min_experience_years": 4,
}

# ── Candidate Resumes ─────────────────────────────────────────────────────────
RESUMES_RAW = [
    {"name": "Alice Chen", "text": """
Alice Chen | alice@email.com
SUMMARY: Data scientist with 6 years of experience building ML solutions.
SKILLS: Python, TensorFlow, PyTorch, scikit-learn, pandas, numpy, SQL, Spark,
Docker, AWS, MLOps, Huggingface, FastAPI, Git, Statistics, Algorithms
EXPERIENCE:
Senior Data Scientist – TechCorp (2020 – 2024)
- Built deep learning NLP models achieving 93% accuracy
- Deployed machine learning models on AWS using Docker and Kubernetes
- Led team of 3 junior data scientists
Data Scientist – DataLabs Inc (2018 – 2020)
- Feature engineering and statistical analysis on large datasets
- Used scikit-learn and XGBoost for predictive modelling
EDUCATION: M.Sc. Computer Science (ML), Stanford, 2018
CERTIFICATIONS: AWS Certified ML Specialist 2022"""},

    {"name": "Bob Martinez", "text": """
Bob Martinez | bob@email.com
OBJECTIVE: Junior data scientist seeking growth in ML environment.
SKILLS: Python, pandas, numpy, matplotlib, scikit-learn, SQL, Git, Excel, Tableau
EXPERIENCE:
Data Analyst – RetailCo (2022 – 2024)
- Analysed sales data using Python and pandas; dashboards in Tableau
- Applied basic regression and classification with scikit-learn
- Wrote SQL queries for PostgreSQL
Intern – AnalyticsFirm (2021 – 2022)
- Data cleaning and exploratory analysis; matplotlib and seaborn
EDUCATION: B.Sc. Statistics, UCLA, 2021"""},

    {"name": "Carol Nguyen", "text": """
Carol Nguyen | carol@email.com
PROFILE: Backend engineer transitioning into ML. 5 years Python backend + deep learning.
SKILLS: Python, JavaScript, Node.js, FastAPI, Django, Docker, Kubernetes, AWS, GCP,
PostgreSQL, MongoDB, Redis, TensorFlow, Keras, scikit-learn, pandas, Git, CI/CD,
Microservices, REST, API, Agile, Scrum
EXPERIENCE:
Senior Backend Engineer – CloudApps (2019 – 2024)
- Architected microservices in Python/FastAPI deployed on AWS via Kubernetes
- Integrated MongoDB and PostgreSQL; Redis for caching; CI/CD pipelines
ML Projects (2022 – 2024)
- Fine-tuned BERT for sentiment analysis (88% accuracy)
- Deployed model as REST API on AWS Lambda
EDUCATION: B.Eng. Software Engineering, Georgia Tech, 2019"""},

    {"name": "David Okafor", "text": """
David Okafor
SKILLS: Java, Spring, MySQL, Git, Linux, Agile
EXPERIENCE:
Java Developer – SoftHouse Ltd (2021 – 2023)
- Built enterprise applications with Java and Spring Boot
- Managed MySQL databases; complex SQL queries
EDUCATION: B.Sc. Computer Science, University of Lagos, 2020"""},

    {"name": "Eva Schmidt", "text": """
Eva Schmidt | eva@email.com
SUMMARY: PhD NLP researcher, 4 years academic + industry experience.
SKILLS: Python, R, NLTK, spaCy, Huggingface, Transformers, PyTorch, scikit-learn,
statistics, machine learning, deep learning, algorithms, data structures,
pandas, numpy, SQL, Git, Tableau, Matplotlib
EXPERIENCE:
NLP Research Scientist – AI Lab Berlin (2022 – 2024)
- Published 3 papers on transformer-based language models
- Implemented NLP models using PyTorch and Huggingface
Research Assistant – TU Berlin (2020 – 2022)
- Statistical analysis and machine learning on medical datasets
EDUCATION: PhD Computational Linguistics, TU Berlin; M.Sc. Data Science, Humboldt, 2019"""},
]

print(f"✅ Loaded {len(RESUMES_RAW)} candidate resumes")
print(f"   Job: {JOB['title']}")
print(f"   Required skills ({len(JOB['required_skills'])}): {', '.join(JOB['required_skills'])}")


## 4. Parse All Resumes

In [ ]:
parsed_resumes = [parser.parse(r["text"], r["name"]) for r in RESUMES_RAW]

# Quick inspection
for p in parsed_resumes:
    print(f"📄 {p['name']:15s} | exp={p['experience_years']}yr | skills={len(p['skills'])} | words={p['word_count']}")
    print(f"   Skills: {', '.join(p['skills'][:8])}{'...' if len(p['skills'])>8 else ''}")
    print()


## 5. Scoring Engine
Three weighted components:
- **TF-IDF Cosine Similarity** (45%) — semantic relevance to job description  
- **Skill Match Ratio** (40%) — required + preferred skill coverage  
- **Experience Bonus** (15%) — years vs minimum threshold

In [ ]:
class ScoringEngine:
    WEIGHTS = {"tfidf_similarity": 0.45, "skill_match": 0.40, "experience_bonus": 0.15}

    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                                          min_df=1, max_features=5000)

    def _tfidf_score(self, resume_text, jd_text):
        try:
            matrix = self.vectorizer.fit_transform([jd_text, resume_text])
            return float(cosine_similarity(matrix[0:1], matrix[1:2])[0][0])
        except Exception:
            return 0.0

    def _skill_score(self, resume_skills, required_skills, preferred_skills):
        resume_set = set(s.lower() for s in resume_skills)
        req_set    = set(s.lower() for s in required_skills)
        pref_set   = set(s.lower() for s in preferred_skills)
        m_req      = resume_set & req_set
        m_pref     = resume_set & pref_set
        combined   = 0.7*(len(m_req)/max(len(req_set),1)) + 0.3*(len(m_pref)/max(len(pref_set),1))
        return (round(combined,4), sorted(m_req), sorted(m_pref),
                sorted(req_set - resume_set), sorted(pref_set - resume_set))

    def _experience_score(self, years, min_years):
        if min_years <= 0: return 1.0
        r = years / min_years
        if r >= 1.0: return 1.0
        elif r >= 0.75: return 0.7
        elif r >= 0.5: return 0.4
        return 0.1

    def score(self, parsed_resume, job):
        tfidf = self._tfidf_score(parsed_resume["clean_text"], job["description"])
        ss, m_req, m_pref, miss_req, miss_pref = self._skill_score(
            parsed_resume["skills"], job["required_skills"], job["preferred_skills"])
        es = self._experience_score(parsed_resume["experience_years"], job["min_experience_years"])
        total = self.WEIGHTS["tfidf_similarity"]*tfidf + self.WEIGHTS["skill_match"]*ss + self.WEIGHTS["experience_bonus"]*es
        return {
            "name": parsed_resume["name"],
            "total_score":       round(total * 100, 2),
            "tfidf_similarity":  round(tfidf  * 100, 2),
            "skill_match_score": round(ss     * 100, 2),
            "experience_score":  round(es     * 100, 2),
            "experience_years":  parsed_resume["experience_years"],
            "matched_required":  m_req, "matched_preferred": m_pref,
            "missing_required":  miss_req, "missing_preferred": miss_pref,
            "total_skills_found": len(parsed_resume["skills"]),
            "skill_gap_count":   len(miss_req),
        }

engine = ScoringEngine()
scores = [engine.score(p, JOB) for p in parsed_resumes]
print("✅ Scoring complete")


## 6. Candidate Ranking

In [ ]:
GRADE_MAP = [(85,"A+","Excellent Match"),(75,"A","Strong Match"),(65,"B+","Good Match"),
             (55,"B","Moderate Match"),(40,"C","Weak Match"),(0,"D","Poor Match")]

def grade(score):
    for t, g, l in GRADE_MAP:
        if score >= t:
            return g, l
    return "D", "Poor Match"

for s in scores:
    s["grade"], s["fit_label"] = grade(s["total_score"])

df = (pd.DataFrame(scores)
        .sort_values("total_score", ascending=False)
        .reset_index(drop=True))
df.index += 1
df.index.name = "rank"

# Display ranking summary
display_cols = ["name","total_score","grade","fit_label","skill_match_score",
                "tfidf_similarity","experience_years","skill_gap_count"]
print("\n🏆 CANDIDATE RANKING\n")
print(df[display_cols].to_string())


## 7. Skill Gap Analysis

In [ ]:
print("\n📊 SKILL GAP REPORT\n" + "="*60)
for rank, row in df.iterrows():
    g, l = row["grade"], row["fit_label"]
    print(f"\n#{rank} {row['name']}  |  Score: {row['total_score']:.1f}/100  [{g}] {l}")
    print(f"   Experience  : {row['experience_years']} yrs")
    if row["matched_required"]:
        print(f"   ✅ Required  : {', '.join(row['matched_required'])}")
    if row["missing_required"]:
        print(f"   ❌ Missing   : {', '.join(row['missing_required'])}")
    if row["matched_preferred"]:
        print(f"   ⭐ Preferred  : {', '.join(row['matched_preferred'])}")


## 8. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Resume Screening Dashboard — Senior Data Scientist", fontsize=16, fontweight="bold", y=1.01)
colors = ["#2ecc71","#27ae60","#f39c12","#e67e22","#e74c3c"]

# ── Plot 1: Total Score Bar Chart ─────────────────────────────────────────────
ax1 = axes[0, 0]
bars = ax1.barh(df["name"][::-1], df["total_score"][::-1], color=colors[::-1], edgecolor="white", height=0.6)
ax1.set_xlabel("Score (out of 100)")
ax1.set_title("Overall Candidate Score", fontweight="bold")
ax1.axvline(x=75, color="navy", linestyle="--", alpha=0.5, label="Strong Match (75)")
ax1.axvline(x=55, color="orange", linestyle="--", alpha=0.5, label="Moderate (55)")
ax1.legend(fontsize=8)
for bar, (_, row) in zip(bars[::-1], df.iterrows()):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"{row['total_score']:.1f}  [{row['grade']}]", va="center", fontsize=9)
ax1.set_xlim(0, 115)

# ── Plot 2: Score Breakdown Stacked Bar ───────────────────────────────────────
ax2 = axes[0, 1]
x = np.arange(len(df))
w = 0.25
ax2.bar(x - w, df["tfidf_similarity"],  width=w, label="Semantic (45%)", color="#3498db")
ax2.bar(x,     df["skill_match_score"], width=w, label="Skills (40%)",   color="#2ecc71")
ax2.bar(x + w, df["experience_score"],  width=w, label="Experience (15%)", color="#e67e22")
ax2.set_xticks(x)
ax2.set_xticklabels([n.split()[0] for n in df["name"]], rotation=15)
ax2.set_ylabel("Component Score (0-100)")
ax2.set_title("Score Breakdown by Component", fontweight="bold")
ax2.legend(fontsize=8)
ax2.set_ylim(0, 120)

# ── Plot 3: Skill Match Heatmap ───────────────────────────────────────────────
ax3 = axes[1, 0]
all_req = JOB["required_skills"]
heat_data = []
for _, row in df.iterrows():
    matched = set(row["matched_required"])
    heat_data.append([1 if s in matched else 0 for s in all_req])
heat_df = pd.DataFrame(heat_data, index=[n.split()[0] for n in df["name"]], columns=all_req)
sns.heatmap(heat_df, ax=ax3, cmap="RdYlGn", linewidths=0.5,
            cbar_kws={"label": "Matched"}, annot=True, fmt="d")
ax3.set_title("Required Skills Coverage", fontweight="bold")
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=45, ha="right", fontsize=8)

# ── Plot 4: Skill Gap Count vs Experience ─────────────────────────────────────
ax4 = axes[1, 1]
sc = ax4.scatter(df["experience_years"], df["total_score"],
                 s=df["skill_match_score"]*5+50, c=df["skill_gap_count"],
                 cmap="RdYlGn_r", edgecolors="black", linewidth=0.8, alpha=0.85)
for _, row in df.iterrows():
    ax4.annotate(row["name"].split()[0], (row["experience_years"], row["total_score"]),
                 textcoords="offset points", xytext=(6, 4), fontsize=8)
plt.colorbar(sc, ax=ax4, label="Skill Gaps (missing required)")
ax4.set_xlabel("Years of Experience")
ax4.set_ylabel("Total Score")
ax4.set_title("Experience vs Score (bubble = skill match)", fontweight="bold")
ax4.axhline(y=75, color="green", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("screening_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard saved to screening_dashboard.png")


## 9. Screen Your Own Resume

In [ ]:
def screen_resume(candidate_name, resume_text, job=JOB):
    """Screen a single resume against the job description."""
    parsed = parser.parse(resume_text, candidate_name)
    scored = engine.score(parsed, job)
    g, l = grade(scored["total_score"])
    scored["grade"] = g
    scored["fit_label"] = l

    print(f"\n{'='*55}")
    print(f"  SCREENING RESULT: {candidate_name}")
    print(f"{'='*55}")
    print(f"  Overall Score  : {scored['total_score']:.1f}/100  [{g}] {l}")
    print(f"  Semantic Sim   : {scored['tfidf_similarity']:.1f}")
    print(f"  Skill Match    : {scored['skill_match_score']:.1f}")
    print(f"  Experience     : {scored['experience_years']} yrs")
    print(f"  Skills Found   : {scored['total_skills_found']}")
    if scored["matched_required"]:
        print(f"  ✅ Req Matched : {', '.join(scored['matched_required'])}")
    if scored["missing_required"]:
        print(f"  ❌ Missing     : {', '.join(scored['missing_required'])}")
    if scored["matched_preferred"]:
        print(f"  ⭐ Pref Matched: {', '.join(scored['matched_preferred'])}")
    print(f"{'='*55}\n")
    return scored

# ── Example: try your own resume text here ───────────────────────────────────
my_resume = """
Your Name
SKILLS: Python, TensorFlow, scikit-learn, SQL, pandas, Docker, AWS
EXPERIENCE:
Data Scientist – Company X (2021 – 2024)
- Built classification models using Python and scikit-learn
- Deployed ML models on AWS; 3 years experience in machine learning
EDUCATION: B.Sc. Computer Science, 2021
"""

result = screen_resume("Your Name", my_resume)


## 10. Summary

### What This System Does
| Feature | Implementation |
|---------|----------------|
| **Text Cleaning** | Regex-based noise removal, section splitting |
| **Skill Extraction** | Pattern matching across 7 technology categories |
| **TF-IDF Similarity** | Bigram TF-IDF with cosine similarity (sklearn) |
| **Skill Matching** | Set-intersection of required + preferred skills |
| **Experience Scoring** | Date-range parsing + heuristic year extraction |
| **Ranking** | Weighted composite score, letter grade |
| **Skill Gap Report** | Per-candidate missing required/preferred skills |
| **Visualisation** | 4-panel matplotlib/seaborn dashboard |

### Potential Enhancements
- 🔍 Named Entity Recognition via spaCy for richer extraction  
- 🤗 Sentence Transformers / BERT embeddings for semantic matching  
- 📊 Bias detection and fairness auditing  
- 🌐 Web scraping for live job postings  
- 🗃️ Database integration for bulk screening  

---
**Repository:** `FUTURE_ML_03`  
**Track:** Machine Learning
